# Week 3 — Functions & First Contact with Data: Reading Real Data with Python
## Phase 2a Python | PORA Academy Cohort 7

By the end of this session, you will be able to:
- Open and read CSV files using `open()` and the `csv` module
- Build a list of dicts from a CSV file
- Apply functions to CSV data
- Understand why pandas will make all of this easier

## The Business Problem

So far we have *typed* Olist values into our notebooks by hand — a single order id here, one price there. That works for learning, but the real Olist orders dataset has **99,441 rows** sitting in a file called `olist_orders_dataset.csv`. Nobody is going to type 99,441 orders into Python by hand.

Today we open that file for the first time and read it with pure Python. By the end of this notebook you will have loaded all 99,441 orders, classified every delivery, and counted every status — using nothing but `open()`, the `csv` module, functions, and a dictionary. It will take real effort.

That effort is the point. Next week we meet **pandas**, and the 25+ lines of code you write today will collapse into about 3. You will appreciate the magic far more once you have done it the hard way.

## Setup — Mounting Drive and Locating the File

Unlike the next few weeks, we are **not** loading pandas today. We only need three things from the standard library and Colab: the `csv` module to parse the file, `os` to build a safe file path, and `drive` to reach the dataset on your Google Drive.

Run the cell below. When Colab asks for permission to access your Drive, click through and allow it. After mounting, `olist_path` points to the folder that contains all the Olist CSV files.

In [ ]:
import csv
from google.colab import drive
import os

# Mount Google Drive so we can reach the dataset
drive.mount('/content/drive')

# Folder that holds all the Olist CSV files
olist_path = '/content/drive/MyDrive/olist-data'

# Build the full path to the orders file (os.path.join handles the slash for us)
orders_filepath = os.path.join(olist_path, 'olist_orders_dataset.csv')
print("Orders file path:", orders_filepath)
# expected: Orders file path: /content/drive/MyDrive/olist-data/olist_orders_dataset.csv

## Concept 1 — Reading a CSV with `open()` and the `csv` module

A CSV file ("comma-separated values") is just a plain text file where each line is a row and commas separate the columns. The very first line is usually the **header** — the column names. Python can open this file like a book and read it one line at a time.

Think of `open()` as walking up to a filing cabinet and pulling out a drawer. The `csv.DictReader` is your assistant who reads each row aloud and hands it to you as a labelled dictionary — `{'order_id': '...', 'order_status': 'delivered', ...}` — so you never have to count comma positions yourself. The `with` keyword is the polite part: it guarantees the drawer gets closed again when you are done, even if something goes wrong.

We will read every row just to count them and confirm the file is the size we expect.

In [ ]:
# Open the orders file and read it row by row with DictReader

row_count = 0
with open(orders_filepath, 'r') as f:
    reader = csv.DictReader(f)        # reads the header automatically
    print("Headers:", reader.fieldnames)
    # expected: Headers: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
    for row in reader:
        row_count += 1

print(f"Total rows: {row_count:,}")
# expected: Total rows: 99,441

## Concept 2 — Building a Reusable `load_orders()` Function

Counting rows is nice, but to *analyse* the data we need to keep it. The natural Python structure is a **list of dictionaries**: one dictionary per order, all collected into one list. Once loaded, `orders[0]` is the first order and `orders[0]['order_status']` is its status.

Rather than rewrite the open/read loop every time, we wrap it in a function called `load_orders()`. This is exactly the function-writing skill from Wednesday, now pointed at a real file. We also give it a `limit` parameter (a default argument) so that while we are still exploring we can load just the first 1,000 rows instead of waiting on all 99,441 — a small, fast sample.

Notice how the function reads almost like an English sentence: open the file, for each row, append it to the list, return the list.

In [ ]:
def load_orders(filepath, limit=None):
    """
    Load orders CSV into a list of dicts.

    Args:
        filepath: path to CSV file
        limit: max rows to load (None = all rows)
    Returns:
        list of dicts
    """
    orders = []
    with open(filepath, 'r') as f:
        reader = csv.DictReader(f)
        for i, row in enumerate(reader):
            if limit and i >= limit:
                break
            orders.append(row)
    return orders


# Explore with a fast 1,000-row sample first
orders_sample = load_orders(orders_filepath, limit=1000)
print(f"Loaded: {len(orders_sample)} orders")
# expected: Loaded: 1000 orders
print(f"Status of first order: {orders_sample[0]['order_status']}")
# expected: Status of first order: delivered

### Counting statuses in the sample

Because each order is a dictionary, we can pull one field out of every order and count how often each value appears. `collections.Counter` is built for exactly this — feed it values and it tallies them. We sort the result with `.most_common()` so the biggest categories come first.

In [ ]:
from collections import Counter

# Count order_status across the 1,000-row sample
status_counts = Counter(o['order_status'] for o in orders_sample)
for status, count in status_counts.most_common():
    print(f"  {status}: {count}")
# expected: delivered is by far the largest status in the sample

## Concept 3 — Applying a Function to All 99,441 Rows

Now we go full scale. We will load **every** order, then run a classification function on each one to label its delivery speed as Fast, Normal, Slow, Not Delivered, or Unknown.

The function works directly on the **raw string values** that came out of the CSV — remember, everything `DictReader` gives us is text, including the dates. So our function parses the date strings itself with `datetime.strptime`, and uses `try/except` (Wednesday's skill) to gracefully handle any row whose dates do not parse. An empty delivery date means the order was never delivered, so we catch that case first.

This is the big moment. We load all the data, loop over all of it, apply a function to each row, and aggregate the results with `Counter`.

In [ ]:
from collections import Counter
from datetime import datetime


def classify_delivery_days_str(purchase_str, delivery_str):
    """Works directly on the raw string values from the CSV."""
    if not delivery_str:                 # empty string => never delivered
        return "Not Delivered"
    try:
        p = datetime.strptime(purchase_str[:19], "%Y-%m-%d %H:%M:%S")
        d = datetime.strptime(delivery_str[:19], "%Y-%m-%d %H:%M:%S")
        days = (d - p).days
        if days <= 7:
            return "Fast"
        elif days <= 14:
            return "Normal"
        else:
            return "Slow"
    except Exception:
        return "Unknown"


# Load ALL orders this time (limit=None)
orders = load_orders(orders_filepath)
print(f"Total: {len(orders):,}")
# expected: Total: 99,441

# Apply the function to every single order and tally the result
delivery_counts = Counter()
for order in orders:
    speed = classify_delivery_days_str(
        order['order_purchase_timestamp'],
        order['order_delivered_customer_date']
    )
    delivery_counts[speed] += 1

for speed, count in delivery_counts.most_common():
    pct = count / len(orders) * 100
    print(f"  {speed}: {count:,} ({pct:.1f}%)")
# expected: counts sum to 99,441 across Fast / Normal / Slow / Not Delivered / Unknown

## Going Deeper — Everything from a CSV is a String

Here is the gotcha that catches almost everyone on their first day of file reading: **`DictReader` returns every value as a string, even numbers and dates.** The empty cells in the file (like a missing delivery date) come back as the empty string `''`, not as `None` and not as a number.

That is why our delivery function had to call `datetime.strptime()` to turn text into a real date before subtracting, and why we checked `if not delivery_str` to detect blanks. If you forget this and try to do maths on a CSV value directly, Python will either crash or give you nonsense. Let's prove the point by inspecting the actual types.

In [ ]:
# Look at the raw types coming out of the CSV
sample_order = orders[0]
print("purchase timestamp value:", repr(sample_order['order_purchase_timestamp']))
print("type of that value:", type(sample_order['order_purchase_timestamp']).__name__)
# expected: type of that value: str

# Count orders whose delivery date is the EMPTY STRING (never delivered)
no_delivery = sum(1 for o in orders if o['order_delivered_customer_date'] == "")
print(f"Orders with a blank delivery date: {no_delivery:,}")
# expected: Orders with a blank delivery date: 2,965

## Common Mistakes

Two errors trip up almost every beginner reading files for the first time. The first is forgetting that CSV values are strings and trying to do arithmetic on them. The second is opening a file without `with`, which can leave the file handle dangling. Study the wrong-then-right pairs below.

In [ ]:
# ── COMMON MISTAKE 1: treating a CSV value as a number ───────────────
# WRONG — this raises TypeError because the value is a string:
# total = orders[0]['order_purchase_timestamp'] + 1
#   TypeError: can only concatenate str (not "int") to str

# CORRECT — parse the string into the type you actually need first:
purchase = datetime.strptime(orders[0]['order_purchase_timestamp'][:19], "%Y-%m-%d %H:%M:%S")
print("Parsed into a real datetime:", purchase)
# expected: Parsed into a real datetime: 2017-10-02 10:56:33


# ── COMMON MISTAKE 2: opening a file without `with` ──────────────────
# RISKY — you must remember to close it yourself, and a crash leaks the handle:
# f = open(orders_filepath, 'r')
# reader = csv.DictReader(f)
# ... if anything errors here, f is never closed ...

# CORRECT — `with` always closes the file for you, even on error:
with open(orders_filepath, 'r') as f:
    first_line = f.readline().strip()
print("Header line read safely:", first_line[:40], "...")
# expected: Header line read safely: order_id,customer_id,order_status,order ...

## Mini-Challenge — Count the Delivered Orders

⏱ ~5 minutes

Using the full `orders` list (all 99,441 rows already loaded above), count how many orders have an `order_status` of exactly `"delivered"`.

**Expected output:**
```
Delivered orders: 96,478
```

Fill in the blanks below. You only need a loop or a comprehension and a counter — both were used earlier in this notebook.

In [ ]:
# `orders` is already loaded (99,441 rows) — do not reload it.

# Your code here: count orders where order_status == "delivered"
delivered_count = 0  # replace this with your counting logic

# Your code here: print the result formatted with a thousands separator
# print(f"Delivered orders: {delivered_count:,}")

# Target: Delivered orders: 96,478

## Thursday Group Exercise: Analyse the Orders Dataset

Using the loaded orders list (all 99,441 rows):

```python
# Task 1: Count how many orders have a blank order_delivered_customer_date
#    (Hint: check if the value is an empty string "")
#    Expected: ~2,965 rows have no delivery date

# Task 2: Write a function count_by_status(orders_list) that returns a dict
#    of {status: count} for all statuses in the list
#    Verify: delivered should be 96,478

# Task 3: Using a list comprehension, create a list of all unique order statuses
#    How many distinct statuses are there?

# Task 4: From the orders list, filter to only "canceled" orders
#    How many are there? Expected: 625
```

In [ ]:
# `orders` is already loaded (all 99,441 rows). Solve each task below.

# Task 1: count orders with a blank order_delivered_customer_date  (expected ~2,965)
# Your code here


# Task 2: write count_by_status(orders_list) -> {status: count}   (delivered == 96,478)
def count_by_status(orders_list):
    # Your code here
    pass


# Task 3: list comprehension of all UNIQUE order statuses, then how many?
# Your code here


# Task 4: filter to only "canceled" orders, then count them          (expected 625)
# Your code here

## Session Summary

| Concept | Key Syntax | Example |
|---|---|---|
| Open a file safely | `with open(path) as f:` | `with open(orders_filepath, 'r') as f:` |
| Read rows as dicts | `csv.DictReader(f)` | `reader = csv.DictReader(f)` |
| Build a list of dicts | `list.append(row)` | `orders.append(row)` |
| Reusable loader | `def load_orders(filepath, limit=None):` | `orders = load_orders(orders_filepath)` |
| Tally values | `collections.Counter` | `Counter(o['order_status'] for o in orders)` |
| Parse a date string | `datetime.strptime(s, fmt)` | `datetime.strptime(s[:19], "%Y-%m-%d %H:%M:%S")` |

Today we read all **99,441 orders** with pure Python. It took `open()`, a `csv.DictReader`, a hand-written `load_orders()` function, a parsing function, a `try/except`, and a `Counter` — well over 25 lines of code to load and classify the data.

**Here is the punchline:** next week, pandas does the loading in a *single* line — `orders = pd.read_csv(...)` — and the counting in one more — `orders['order_status'].value_counts()`. Everything you sweated through today, pandas does in about 3 lines. You now know exactly what pandas is doing under the hood, which is why you will trust it.

---
**Coming up Wednesday**: Welcome to pandas — loading the entire Olist dataset in one line and exploring DataFrames.